# Did the extra views turn into likes?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** No. More views, but not more likes, and no extra organic arguing in the replies either.



# Podolak-Inspired Analysis

**Inspired by:** Podolak et al. 2024 — *LLM Generated Responses to Mitigate the Impact of Hate Speech* (EMNLP 2024 Findings)

Podolak et al. run a Twitter A/B test of LLM-generated CN replies on hate-speech tweets and find **>20% suppression of engagement** — the opposite of our anti-suppression result.
This notebook replicates their analytical structure to enable a direct design-level comparison.

## Analyses implemented

| # | Analysis | What it adds |
|---|----------|--------------|
| 1 | **Engagement quality ratio** E(i) = Δlikes / (Δviews+1) | Do the extra views attracted by CNs convert to endorsements? |
| 2 | **Reply intensity ratio** R(i) = Δcomments / (Δviews+1) | Do CN-exposed tweets attract more organic discussion per view? |
| 3 | **Original post vs. reply subgroup** (is_comment) | Podolak finds opposite signs for original tweets vs. replies |
| 4 | **Baseline virality threshold analysis** | Podolak's effect holds at ≥10 views but vanishes at ≥100 — does ours? |

**Sign convention:** `rank_biserial_r(trt, ctrl)` → r < 0 = Treatment grew more = **anti-suppression** (our main finding).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then take the field-experiment folder inside it.
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
BASE_DIR = _root / "field-experiment"
if not (BASE_DIR / "data").is_dir():
    raise RuntimeError(
        "Could not locate the field-experiment folder from " + str(Path.cwd()) +
        ". Run this notebook from inside the cloned repository."
    )
DATA_DIR   = BASE_DIR / 'data'
FEAT_DIR   = BASE_DIR / 'data'
OUT_DIR    = BASE_DIR / 'outputs' / '05_engagement_quality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_CSV  = FEAT_DIR / 'tweet_features.csv'
MONITORING_XL = DATA_DIR / 'Tweet Monitoring.xlsx'
TREATMENT_XL  = DATA_DIR / 'Treatment_Group.xlsx'

METRICS      = ['Views', 'Likes', 'Shares']
MAIN_WINDOW  = 13
N_BOOTSTRAP  = 5000
RANDOM_SEED  = 42

print('Config loaded.')

In [ ]:
# ── Helper functions ──────────────────────────────────────────

def rank_biserial_r(trt, ctrl):
    U, _ = stats.mannwhitneyu(trt, ctrl, alternative='two-sided')
    return 1 - (2 * U) / (len(trt) * len(ctrl))

def bootstrap_ci(trt, ctrl, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    boots = [
        rank_biserial_r(
            rng.choice(trt,  size=len(trt),  replace=True),
            rng.choice(ctrl, size=len(ctrl), replace=True),
        )
        for _ in range(n_boot)
    ]
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def compute_effect(trt, ctrl, min_n=20):
    trt = np.asarray(trt.dropna() if hasattr(trt, 'dropna') else trt)
    ctrl = np.asarray(ctrl.dropna() if hasattr(ctrl, 'dropna') else ctrl)
    if len(trt) < min_n or len(ctrl) < min_n:
        return None
    _, p = stats.mannwhitneyu(trt, ctrl, alternative='two-sided')
    r = rank_biserial_r(trt, ctrl)
    lo, hi = bootstrap_ci(trt, ctrl)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('.' if p < 0.10 else '')))
    return {'r': r, 'ci_lo': lo, 'ci_hi': hi, 'p': p,
            'n_trt': len(trt), 'n_ctrl': len(ctrl), 'sig': sig}

print('Helpers defined.')

---
## 1. Data Loading

Same pipeline: load tweet_features.csv, pivot Tweet Monitoring.xlsx, compute growth DVs.
Additionally load `Treatment_Group.xlsx` for `Num_of_CNs` (needed to compute adjusted reply ratio R_adj).

In [ ]:
# ── Parse monitoring dates (domain-aware: Dec=2025, Jan=2026) ─────────────────
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    date_str = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if '-' in date_str and date_str[:4].isdigit():
        parts = date_str.split('-')
        year = int(parts[0]); num1 = int(parts[1]); num2 = int(parts[2].split()[0])
        if is_valid(year, num1): month, day = num1, num2
        elif is_valid(year, num2): month, day = num2, num1
        else: month, day = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    elif '/' in date_str:
        parts = date_str.split('/')
        num1 = int(parts[0]); num2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, num2): day, month = num1, num2
        elif is_valid(year, num1): month, day = num1, num2
        else: day, month = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    else:
        return pd.to_datetime(date_val, errors='coerce')

# ── Load tweet features ────────────────────────────────────────────────────────
feat = pd.read_csv(FEATURES_CSV)
print(f'tweet_features.csv: {feat.shape}')
print(f"Group distribution: {feat['Group'].value_counts().to_dict()}")
print(f"is_comment values:  {feat['is_comment'].value_counts().to_dict()}")

# ── Load and pivot monitoring ──────────────────────────────────────────────────
monitoring = pd.read_excel(MONITORING_XL)
monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Sample_Date']           = monitoring['Sample_Date'].apply(parse_mixed_date)
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

pivoted = monitoring.pivot_table(
    index='URL', columns='Day',
    values=['Views', 'Likes', 'Comments', 'Shares'], aggfunc='first'
)
pivoted.columns = [f'{m}_Day{d}' for m, d in pivoted.columns]
pivoted = pivoted.reset_index()
print(f'Pivoted monitoring: {pivoted.shape}')

# ── Merge ─────────────────────────────────────────────────────────────────────
df = feat.merge(pivoted, on='URL', how='left')
print(f'After merge: {df.shape}')

# ── Fill Day0 NaN for Likes/Shares/Comments with 0 ────────────────────────────
for metric in ['Likes', 'Shares', 'Comments']:
    col = f'{metric}_Day0'
    if col in df.columns:
        df[col] = df[col].fillna(0)

# ── Compute growth DVs ─────────────────────────────────────────────────────────
for metric in METRICS + ['Comments']:
    d0 = f'{metric}_Day0'
    dN = f'{metric}_Day{MAIN_WINDOW}'
    if d0 in df.columns and dN in df.columns:
        df[f'{metric}_Growth_{MAIN_WINDOW}d'] = (df[dN] - df[d0]) / (df[d0] + 1) * 100

# ── Log-baseline for regressions ──────────────────────────────────────────────
for metric in METRICS:
    d0 = f'{metric}_Day0'
    if d0 in df.columns:
        df[f'log_baseline_{metric}'] = np.log1p(df[d0])

df['is_treatment'] = (df['Group'] == 'Treatment').astype(int)

print('\nGrowth DV coverage:')
for metric in METRICS + ['Comments']:
    col = f'{metric}_Growth_{MAIN_WINDOW}d'
    if col in df.columns:
        print(f'  {col:<30} {df[col].notna().sum()} / {len(df)}')

In [ ]:
# ── Load Num_of_CNs from Treatment_Group.xlsx ─────────────────────────────────
trt_xl = pd.read_excel(TREATMENT_XL, usecols=['URL', 'Num_of_CNs'])
trt_xl = trt_xl.dropna(subset=['URL'])
trt_xl['Num_of_CNs'] = pd.to_numeric(trt_xl['Num_of_CNs'], errors='coerce').fillna(0).astype(int)
print(f'Treatment_Group rows with Num_of_CNs: {len(trt_xl)}')
print(f'Num_of_CNs:\n{trt_xl["Num_of_CNs"].describe()}')

df = df.merge(trt_xl, on='URL', how='left')
df['Num_of_CNs'] = df['Num_of_CNs'].fillna(0).astype(int)  # 0 for Control

print(f'\nNum_of_CNs by group (should be 0 for all Control):')
print(df.groupby('Group')['Num_of_CNs'].describe())

In [ ]:
# ── Compute absolute deltas and engagement ratios ─────────────────────────────
for metric in METRICS + ['Comments']:
    d0 = f'{metric}_Day0'
    dN = f'{metric}_Day{MAIN_WINDOW}'
    if d0 in df.columns and dN in df.columns:
        df[f'Delta_{metric}'] = df[dN] - df[d0]

# E(i) = Δlikes / (Δviews + 1)  — engagement quality ratio
# Adding 1 to denominator avoids division by zero for stagnant tweets.
# A tweet that gained 0 views and 0 likes gets E = 0/(0+1) = 0.
df['E_ratio'] = df['Delta_Likes'] / (df['Delta_Views'] + 1)

# R_raw(i) = Δcomments / (Δviews + 1) — includes CN replies for Treatment
# Note: each CN reply mechanically adds ≥1 to Comments for Treatment tweets,
# so R_raw is confounded. R_adj subtracts Num_of_CNs from the delta.
df['R_ratio_raw'] = df['Delta_Comments'] / (df['Delta_Views'] + 1)
df['Delta_Comments_adj'] = df['Delta_Comments'] - df['Num_of_CNs']
df['R_ratio_adj'] = df['Delta_Comments_adj'] / (df['Delta_Views'] + 1)

print('Engagement ratios computed.')

ctrl = df[df['Group'] == 'Control']
trt  = df[df['Group'] == 'Treatment']

print(f'\n--- E(i) summary by group ---')
for grp, grp_df in [('Control', ctrl), ('Treatment', trt)]:
    vals = grp_df['E_ratio'].dropna()
    print(f'  {grp}: median={np.median(vals):.4f}  mean={np.mean(vals):.4f}  '
          f'p25={np.percentile(vals,25):.4f}  p75={np.percentile(vals,75):.4f}')

print(f'\n--- R_raw(i) summary by group ---')
for grp, grp_df in [('Control', ctrl), ('Treatment', trt)]:
    vals = grp_df['R_ratio_raw'].dropna()
    print(f'  {grp}: median={np.median(vals):.4f}  mean={np.mean(vals):.4f}')

print(f'\n--- R_adj(i) summary by group (Treatment adjusted for CN reply count) ---')
for grp, grp_df in [('Control', ctrl), ('Treatment', trt)]:
    vals = grp_df['R_ratio_adj'].dropna()
    print(f'  {grp}: median={np.median(vals):.4f}  mean={np.mean(vals):.4f}')

---
## 2. Analysis 1 — Engagement Quality Ratio E(i) = Δlikes / (Δviews + 1)

**Motivation from Podolak 2024:** They find that LLM-generated CN replies suppress both raw impressions and the ratio of likes-to-impressions,
suggesting that the residual audience engaging with CN-tagged tweets is less endorsing than the organic audience.

Our main result shows **views went up** in Treatment but **likes did not**. E(i) makes this gap explicit:
if E(Treatment) < E(Control), the additional views attracted by CNs are lower-quality attention.

**Expected direction:** E(Treatment) < E(Control) → r > 0 (Control E exceeds Treatment E).

In [ ]:
# ── Winsorize E_ratio at 99th percentile (extreme ratios from near-zero view changes)
p99_e = df['E_ratio'].quantile(0.99)
ctrl_e_raw = ctrl['E_ratio'].dropna()
trt_e_raw  = trt['E_ratio'].dropna()
ctrl_e = ctrl_e_raw.clip(upper=p99_e).values
trt_e  = trt_e_raw.clip(upper=p99_e).values

print(f'E_ratio 99th percentile winsorize cap: {p99_e:.4f}')
print(f'N after winsorize: Control={len(ctrl_e)}, Treatment={len(trt_e)}')
print()

eff_e = compute_effect(trt_e, ctrl_e)
print('=' * 70)
print('ENGAGEMENT QUALITY RATIO  E(i) = Δlikes / (Δviews + 1)')
print('=' * 70)
print(f'  Control median E:   {np.median(ctrl_e):.4f}')
print(f'  Treatment median E: {np.median(trt_e):.4f}')
print()
print(f'  Rank-biserial r = {eff_e["r"]:+.4f}  (95% CI [{eff_e["ci_lo"]:.4f}, {eff_e["ci_hi"]:.4f}])')
print(f'  MWU p = {eff_e["p"]:.4f}  {eff_e["sig"]}')
print()
if eff_e['r'] > 0:
    print('  ➔ Control E > Treatment E: CNs attract lower-quality attention (fewer likes per view)')
elif eff_e['r'] < 0:
    print('  ➔ Treatment E > Control E: CNs attract higher-quality attention (more likes per view)')
else:
    print('  ➔ No difference in engagement quality')

print()
print('  Podolak 2024: LLM-generated CNs suppress both impressions AND quality ratio')
print('  Our finding: Views suppressed in opposite direction (anti-suppression);')
print('  E(i) tests whether the extra views convert to likes or are empty attention.')

In [ ]:
# ── Plot: E(i) distribution by group ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: Box plots
ax = axes[0]
bp = ax.boxplot(
    [ctrl_e, trt_e],
    labels=['Control', 'Treatment'],
    patch_artist=True,
    medianprops={'color': 'black', 'linewidth': 2},
    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.3}
)
bp['boxes'][0].set_facecolor('#4C72B0')
bp['boxes'][1].set_facecolor('#DD8452')
ax.set_ylabel('E(i) = Δlikes / (Δviews + 1)', fontsize=10)
ax.set_title('Panel A: Engagement Quality Ratio by Group', fontsize=11, fontweight='bold')
ax.text(0.98, 0.98, f'r = {eff_e["r"]:+.3f}\np = {eff_e["p"]:.3f} {eff_e["sig"]}',
        transform=ax.transAxes, va='top', ha='right', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.grid(axis='y', alpha=0.3)

# Panel B: Histogram overlay
ax = axes[1]
clip_val = min(p99_e, 1.0)  # cap histogram at 1 for readability
for vals, label, color in [(ctrl_e, 'Control', '#4C72B0'), (trt_e, 'Treatment', '#DD8452')]:
    ax.hist(np.clip(vals, -clip_val, clip_val), bins=40, alpha=0.5,
            label=label, color=color, density=True)
ax.axvline(np.median(ctrl_e), color='#4C72B0', linestyle='--', linewidth=2, label=f'Control median={np.median(ctrl_e):.3f}')
ax.axvline(np.median(trt_e),  color='#DD8452', linestyle='--', linewidth=2, label=f'Treatment median={np.median(trt_e):.3f}')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('E(i) = Δlikes / (Δviews + 1)  [clipped at ±1]', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Panel B: E(i) Distribution Overlay', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'e_ratio_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# -- E(i) diagnostic: floor effect & restricted to delta_likes > 0 --
#
# The median E = 0 in both groups because most tweets gain zero likes.
# Part 1: quantify the floor effect.
# Part 2: re-run E(i) restricted to tweets where Delta_Likes > 0.
#
# CAUTION: conditioning on Delta_Likes > 0 is conditioning on a post-treatment
# outcome. If CNs caused some Treatment tweets to gain likes they otherwise
# would not have, the two subsamples differ in composition. Treat as descriptive.

print("=" * 70)
print("DIAGNOSTIC: Likes floor effect")
print("=" * 70)

for grp, grp_df in [("Control", ctrl), ("Treatment", trt)]:
    n_total = len(grp_df["Delta_Likes"].dropna())
    n_pos   = (grp_df["Delta_Likes"] > 0).sum()
    n_zero  = (grp_df["Delta_Likes"] == 0).sum()
    n_neg   = (grp_df["Delta_Likes"] < 0).sum()
    print(f"  {grp} (n={n_total}): "
          f"delta>0: {n_pos} ({n_pos/n_total*100:.1f}%)  "
          f"delta=0: {n_zero} ({n_zero/n_total*100:.1f}%)  "
          f"delta<0: {n_neg} ({n_neg/n_total*100:.1f}%)")

print()
print("=" * 70)
print("E(i) RESTRICTED TO TWEETS WITH delta_likes > 0")
print("=" * 70)

# Restrict to tweets that gained at least one like
pos_likes = df[df["Delta_Likes"] > 0].copy()
ctrl_pos  = pos_likes[pos_likes["Group"] == "Control"]
trt_pos   = pos_likes[pos_likes["Group"] == "Treatment"]

print(f"  Subsample: Control n={len(ctrl_pos)}, Treatment n={len(trt_pos)}")
print()

p99_e_pos  = pos_likes["E_ratio"].quantile(0.99)
ctrl_e_pos = ctrl_pos["E_ratio"].clip(upper=p99_e_pos).dropna().values
trt_e_pos  = trt_pos["E_ratio"].clip(upper=p99_e_pos).dropna().values

print(f"  99th pct winsorize cap (subsample): {p99_e_pos:.4f}")
print(f"  Control   median E: {np.median(ctrl_e_pos):.4f}   mean E: {np.mean(ctrl_e_pos):.4f}")
print(f"  Treatment median E: {np.median(trt_e_pos):.4f}   mean E: {np.mean(trt_e_pos):.4f}")
print()

if len(ctrl_e_pos) >= 20 and len(trt_e_pos) >= 20:
    eff_e_pos = compute_effect(trt_e_pos, ctrl_e_pos)
    print(f"  Rank-biserial r = {eff_e_pos["r"]:+.4f}  "
          f"(95% CI [{eff_e_pos["ci_lo"]:.4f}, {eff_e_pos["ci_hi"]:.4f}])")
    print(f"  MWU p = {eff_e_pos["p"]:.4f}  {eff_e_pos["sig"]}")
    print()
    if eff_e_pos["r"] > 0:
        print("  => Control E > Treatment E: among tweets that gained likes,")
        print("     CN-exposed tweets attracted fewer likes per view gained.")
    elif eff_e_pos["r"] < 0:
        print("  => Treatment E > Control E: among tweets that gained likes,")
        print("     CN-exposed tweets attracted more likes per view gained.")
    else:
        print("  => No difference in E(i) among tweets that gained likes.")
else:
    print("  Subsample too small for a reliable test.")

---
## 3. Analysis 2 — Reply Intensity Ratio R(i) = Δcomments / (Δviews + 1)

**Motivation from Podolak 2024:** They find replies per impression increase by ~158% for original tweets — CNs attract organic counter-discussion.

**Confound for our data:** Our CN replies are *themselves* comments on the target tweet, so `Delta_Comments` for Treatment tweets mechanically includes the CN reply. We compute two versions:
- **R_raw:** raw Δcomments / (Δviews + 1) — includes all comments including the CN reply itself
- **R_adj:** (Δcomments − Num_of_CNs) / (Δviews + 1) — organic replies only, subtracting the CN count

R_adj is the more meaningful quantity: it tests whether CN-exposed tweets attracted *additional* organic discussion beyond the CN itself.

In [ ]:
# ── Winsorize R ratios at 99th percentile ─────────────────────────────────────
p99_r = df['R_ratio_adj'].quantile(0.99)
ctrl_r_raw = ctrl['R_ratio_raw'].clip(upper=p99_r).dropna().values
trt_r_raw  = trt['R_ratio_raw'].clip(upper=p99_r).dropna().values
ctrl_r_adj = ctrl['R_ratio_adj'].clip(upper=p99_r).dropna().values
trt_r_adj  = trt['R_ratio_adj'].clip(upper=p99_r).dropna().values

eff_r_raw = compute_effect(trt_r_raw, ctrl_r_raw)
eff_r_adj = compute_effect(trt_r_adj, ctrl_r_adj)

print('=' * 70)
print('REPLY INTENSITY RATIO  R(i) = Δcomments / (Δviews + 1)')
print('=' * 70)
print(f'  99th pct winsorize cap: {p99_r:.4f}')
print()
print('  R_RAW (includes CN replies — confounded for Treatment):')
print(f'    Control median: {np.median(ctrl_r_raw):.4f}   Treatment median: {np.median(trt_r_raw):.4f}')
print(f'    r = {eff_r_raw["r"]:+.4f}  (95% CI [{eff_r_raw["ci_lo"]:.4f}, {eff_r_raw["ci_hi"]:.4f}])  p = {eff_r_raw["p"]:.4f} {eff_r_raw["sig"]}')
print()
print('  R_ADJ (Treatment delta adjusted for Num_of_CNs — organic replies only):')
print(f'    Control median: {np.median(ctrl_r_adj):.4f}   Treatment median: {np.median(trt_r_adj):.4f}')
print(f'    r = {eff_r_adj["r"]:+.4f}  (95% CI [{eff_r_adj["ci_lo"]:.4f}, {eff_r_adj["ci_hi"]:.4f}])  p = {eff_r_adj["p"]:.4f} {eff_r_adj["sig"]}')
print()
print('  Podolak 2024: R increases ~158% for original tweets (LLM CN drives more replies)')
print('  Interpretation (R_adj): does receiving a human-crafted CN drive organic counter-replies?')
if eff_r_adj['r'] < 0:
    print('  ➔ Treatment R_adj > Control: CN-exposed tweets attract more organic discussion per view')
elif eff_r_adj['r'] > 0:
    print('  ➔ Control R_adj > Treatment: CN-exposed tweets attract fewer organic replies per view')
else:
    print('  ➔ No difference in organic reply intensity')

In [ ]:
# ── Plot: R(i) raw vs. adjusted comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (c_vals, t_vals, eff, label, ylabel) in zip(axes, [
    (ctrl_r_raw, trt_r_raw, eff_r_raw, 'R_raw (includes CN reply)', 'Δcomments / (Δviews+1) [raw]'),
    (ctrl_r_adj, trt_r_adj, eff_r_adj, 'R_adj (organic replies only)', 'Δcomments_adj / (Δviews+1)'),
]):
    bp = ax.boxplot(
        [c_vals, t_vals],
        labels=['Control', 'Treatment'],
        patch_artist=True,
        medianprops={'color': 'black', 'linewidth': 2},
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.3}
    )
    bp['boxes'][0].set_facecolor('#4C72B0')
    bp['boxes'][1].set_facecolor('#DD8452')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.text(0.98, 0.98, f'r = {eff["r"]:+.3f}\np = {eff["p"]:.3f} {eff["sig"]}',
            transform=ax.transAxes, va='top', ha='right', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Reply Intensity Ratio R(i) by Group', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'r_ratio_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# -- R_adj diagnostic: floor effect & restricted to delta_comments_adj > 0 --
# Same logic as E(i) floor analysis. R_adj = 0 at the median because most
# tweets gain zero organic comments. Restricting to tweets that gained at
# least one organic comment (Delta_Comments_adj > 0) tests whether CN-exposed
# tweets convert views to organic replies more or less efficiently.
#
# CAUTION: same post-treatment selection bias caveat as E(i) analysis.

print("=" * 70)
print("DIAGNOSTIC: Organic comments floor effect")
print("=" * 70)

for grp, grp_df in [("Control", ctrl), ("Treatment", trt)]:
    n_total = len(grp_df["Delta_Comments_adj"].dropna())
    n_pos   = (grp_df["Delta_Comments_adj"] > 0).sum()
    n_zero  = (grp_df["Delta_Comments_adj"] == 0).sum()
    n_neg   = (grp_df["Delta_Comments_adj"] < 0).sum()
    print(f"  {grp} (n={n_total}): "
          f"adj_delta>0: {n_pos} ({n_pos/n_total*100:.1f}%)  "
          f"adj_delta=0: {n_zero} ({n_zero/n_total*100:.1f}%)  "
          f"adj_delta<0: {n_neg} ({n_neg/n_total*100:.1f}%)")

print()
print("=" * 70)
print("R_adj RESTRICTED TO TWEETS WITH delta_comments_adj > 0")
print("=" * 70)

pos_comments = df[df["Delta_Comments_adj"] > 0].copy()
ctrl_pos_c   = pos_comments[pos_comments["Group"] == "Control"]
trt_pos_c    = pos_comments[pos_comments["Group"] == "Treatment"]

print(f"  Subsample: Control n={len(ctrl_pos_c)}, Treatment n={len(trt_pos_c)}")
print()

p99_r_pos   = pos_comments["R_ratio_adj"].quantile(0.99)
ctrl_r_pos  = ctrl_pos_c["R_ratio_adj"].clip(upper=p99_r_pos).dropna().values
trt_r_pos   = trt_pos_c["R_ratio_adj"].clip(upper=p99_r_pos).dropna().values

print(f"  99th pct winsorize cap (subsample): {p99_r_pos:.4f}")
print(f"  Control   median R_adj: {np.median(ctrl_r_pos):.4f}   mean: {np.mean(ctrl_r_pos):.4f}")
print(f"  Treatment median R_adj: {np.median(trt_r_pos):.4f}   mean: {np.mean(trt_r_pos):.4f}")
print()

if len(ctrl_r_pos) >= 20 and len(trt_r_pos) >= 20:
    eff_r_pos = compute_effect(trt_r_pos, ctrl_r_pos)
    print(f"  Rank-biserial r = {eff_r_pos["r"]:+.4f}  "
          f"(95% CI [{eff_r_pos["ci_lo"]:.4f}, {eff_r_pos["ci_hi"]:.4f}])")
    print(f"  MWU p = {eff_r_pos["p"]:.4f}  {eff_r_pos["sig"]}")
    print()
    if eff_r_pos["r"] < 0:
        print("  => Treatment R_adj > Control: among tweets that gained organic comments,")
        print("     CN-exposed tweets attracted more organic replies per view gained.")
    elif eff_r_pos["r"] > 0:
        print("  => Control R_adj > Treatment: among tweets that gained organic comments,")
        print("     CN-exposed tweets attracted fewer organic replies per view gained.")
    else:
        print("  => No difference in R_adj among tweets that gained organic comments.")
else:
    print("  Subsample too small for a reliable test.")

---
## 4. Analysis 3 — Original Post vs. Reply Subgroup (is_comment)

**Motivation from Podolak 2024:** They find **opposite effects** depending on tweet type:
- Original tweets (not replies): CN replies suppress impressions significantly
- Reply tweets (tweets that are themselves replies): CN replies have no significant effect (or positive)

We have `is_comment` in our data (0 = original post, 1 = tweet that is itself a reply/comment).
This subgroup has never been used in any of our analyses — it is a completely new split.

**Key question:** Does our anti-suppression effect differ between original propaganda posts and reply-type propaganda posts?

In [ ]:
# ── Check subgroup sizes ───────────────────────────────────────────────────────
print('is_comment distribution by group:')
print(df.groupby(['Group', 'is_comment']).size().unstack(fill_value=0))
print()
# is_comment = 0 → original post, = 1 → reply/comment
print('Tweet type labels:')
print("  is_comment=0 → original tweet (not a reply to another tweet)")
print("  is_comment=1 → this tweet is itself a reply to another tweet")
print()
print("Note: in our data the 'Type' column was the original source;")
print('is_comment was derived from the tweet's type. Both should agree.')
if 'Type' in df.columns:
    print(df.groupby(['Group', 'Type']).size().unstack(fill_value=0))

In [ ]:
# ── MWU by is_comment subgroup for all three metrics ──────────────────────────
subgroup_results = []

print('=' * 85)
print('SUBGROUP ANALYSIS: is_comment × Group  (rank-biserial r, MWU)')
print('r < 0 = anti-suppression (Treatment grew more);  r > 0 = suppression')
print('=' * 85)

type_labels = {0: 'Original post', 1: 'Reply/comment'}
for ic_val, type_label in type_labels.items():
    sub = df[df['is_comment'] == ic_val]
    ctrl_sub = sub[sub['Group'] == 'Control']
    trt_sub  = sub[sub['Group'] == 'Treatment']
    print(f'\n  {type_label} (is_comment={ic_val})  '
          f'N_ctrl={len(ctrl_sub)}, N_trt={len(trt_sub)}')
    print(f'  {"Metric":<10} {"r":>8} {"CI_lo":>8} {"CI_hi":>8} {"p":>8} {"sig":>5}')
    print(f'  {"-"*52}')
    for metric in METRICS:
        col = f'{metric}_Growth_{MAIN_WINDOW}d'
        eff = compute_effect(trt_sub[col].dropna(), ctrl_sub[col].dropna())
        if eff:
            print(f'  {metric:<10} {eff["r"]:>+8.3f} {eff["ci_lo"]:>8.3f} {eff["ci_hi"]:>8.3f} '
                  f'{eff["p"]:>8.4f} {eff["sig"]:>5}')
            subgroup_results.append({'type_label': type_label, 'is_comment': ic_val,
                                     'metric': metric, **eff})
        else:
            print(f'  {metric:<10}   SKIP (n too small)')

subgroup_df = pd.DataFrame(subgroup_results)

In [ ]:
# ── Forest plot: is_comment × Metric ──────────────────────────────────────────
if len(subgroup_df) == 0:
    print('No results to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    colors = {0: '#4C72B0', 1: '#DD8452'}
    labels = {0: 'Original post (is_comment=0)', 1: 'Reply/comment (is_comment=1)'}

    for ax, metric in zip(axes, METRICS):
        metric_rows = subgroup_df[subgroup_df['metric'] == metric].copy()
        y_positions = range(len(metric_rows))

        for y, (_, row) in zip(y_positions, metric_rows.iterrows()):
            color = colors[row['is_comment']]
            ax.plot([row['ci_lo'], row['ci_hi']], [y, y], color=color, linewidth=2)
            marker = 'D' if row['p'] < 0.05 else 'o'
            ax.plot(row['r'], y, marker=marker, color=color, markersize=8, zorder=5)
            sig_txt = row['sig'] if row['sig'] else ''
            ax.text(row['ci_hi'] + 0.01, y, f" {sig_txt}", va='center', fontsize=9)

        ax.axvline(0, color='black', linewidth=1, linestyle='--')
        ax.set_yticks(list(y_positions))
        ax.set_yticklabels([labels[r['is_comment']] for _, r in metric_rows.iterrows()], fontsize=8)
        ax.set_xlabel('Rank-biserial r\n(← anti-suppression | suppression →)', fontsize=9)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        ax.set_xlim(-0.5, 0.5)

    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0],[0], marker='D', color='gray', markersize=8, linestyle='None', label='p < 0.05'),
        Line2D([0],[0], marker='o', color='gray', markersize=8, linestyle='None', label='p ≥ 0.05'),
    ]
    axes[-1].legend(handles=legend_elems, loc='lower right', fontsize=8)

    plt.suptitle('CN Effect by Tweet Type (original post vs. reply)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'subgroup_is_comment_forest.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')

---
## 5. Analysis 4 — Baseline Virality Threshold Analysis

**Motivation from Podolak 2024:** Their suppression effect holds for tweets with **≥10 initial views** but disappears at **≥100 views**.
The threshold filters to tweets with at least some minimum pre-treatment visibility — eliminating truly dormant tweets where any change is noise.

We have `Views_Day0` as the baseline virality measure. Three thresholds:
- **≥10 views** at Day 0 (low bar — most tweets)
- **≥100 views** at Day 0 (moderate virality)
- **≥500 views** at Day 0 (high virality)

**Key question:** Is our anti-suppression effect concentrated in low-virality or high-virality tweets?

In [ ]:
# ── Describe baseline views distribution ──────────────────────────────────────
print('Views_Day0 distribution:')
print(df['Views_Day0'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))
print()
for thresh in [10, 50, 100, 500, 1000]:
    n = (df['Views_Day0'] >= thresh).sum()
    pct = n / len(df) * 100
    print(f'  ≥{thresh:>5} views at Day0: {n:>5} tweets ({pct:.1f}%)')

In [ ]:
# ── Threshold analysis: effect at increasing minimum baseline views ────────────
thresholds = [0, 10, 100, 500]
threshold_results = []

print('=' * 95)
print('THRESHOLD ANALYSIS: CN effect restricted to tweets with Views_Day0 ≥ threshold')
print('r < 0 = anti-suppression;  Diamond (◆) = p < 0.05')
print('=' * 95)

for thresh in thresholds:
    sub = df[df['Views_Day0'] >= thresh]
    ctrl_sub = sub[sub['Group'] == 'Control']
    trt_sub  = sub[sub['Group'] == 'Treatment']
    label = f'All (≥{thresh})' if thresh == 0 else f'≥{thresh} views'
    print(f'\n  {label}  (N_ctrl={len(ctrl_sub)}, N_trt={len(trt_sub)})')
    print(f'  {"Metric":<10} {"r":>8} {"CI_lo":>8} {"CI_hi":>8} {"p":>8} {"sig":>5}')
    print(f'  {"-"*50}')
    for metric in METRICS:
        col = f'{metric}_Growth_{MAIN_WINDOW}d'
        eff = compute_effect(trt_sub[col].dropna(), ctrl_sub[col].dropna(), min_n=0)
        if eff:
            print(f'  {metric:<10} {eff["r"]:>+8.3f} {eff["ci_lo"]:>8.3f} {eff["ci_hi"]:>8.3f} '
                  f'{eff["p"]:>8.4f} {eff["sig"]:>5}')
            threshold_results.append({'threshold': thresh, 'label': label, 'metric': metric,
                                      'n_ctrl': len(ctrl_sub), 'n_trt': len(trt_sub), **eff})
        else:
            print(f'  {metric:<10}   SKIP (n too small)')

threshold_df = pd.DataFrame(threshold_results)

In [ ]:
# ── Plot: effect size vs. threshold for each metric ───────────────────────────
if len(threshold_df) == 0:
    print('No results to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    thresh_labels = {t: (f'≥{t}' if t > 0 else 'All') for t in thresholds}

    for ax, metric in zip(axes, METRICS):
        rows = threshold_df[threshold_df['metric'] == metric].sort_values('threshold')
        x = range(len(rows))

        ax.errorbar(
            x,
            rows['r'],
            yerr=[rows['r'] - rows['ci_lo'], rows['ci_hi'] - rows['r']],
            fmt='o', color='#2176AE', capsize=5, linewidth=2, markersize=8
        )
        # Highlight significant points
        for xi, (_, row) in zip(x, rows.iterrows()):
            if row['p'] < 0.05:
                ax.plot(xi, row['r'], 'D', color='#C44E52', markersize=10, zorder=5)

        ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_xticks(list(x))
        ax.set_xticklabels(
            [f"{thresh_labels[r['threshold']]}\n(n={r['n_trt']}T/{r['n_ctrl']}C)"
             for _, r in rows.iterrows()],
            fontsize=8
        )
        ax.set_xlabel('Minimum baseline views (Views_Day0)', fontsize=9)
        if ax == axes[0]:
            ax.set_ylabel('Rank-biserial r\n(← anti-suppression | suppression →)', fontsize=9)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(-0.4, 0.4)

    from matplotlib.lines import Line2D
    legend_elems = [
        Line2D([0],[0], marker='D', color='#C44E52', markersize=8, linestyle='None', label='p < 0.05'),
        Line2D([0],[0], marker='o', color='#2176AE', markersize=8, linestyle='None', label='p ≥ 0.05'),
    ]
    axes[-1].legend(handles=legend_elems, loc='lower right', fontsize=8)

    plt.suptitle('CN Effect by Baseline Virality Threshold', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'threshold_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')

---
## 6. Summary Comparison Table vs. Podolak 2024

Side-by-side comparison of our results and Podolak et al.'s for the analyses replicable across both studies.

In [ ]:
# ── Helper string formatters for the summary table ────────────────────────────

def _get_subgroup_str(df_, ic_val, metric):
    rows = df_[(df_['is_comment'] == ic_val) & (df_['metric'] == metric)]
    if len(rows) == 0:
        return 'N/A'
    r = rows.iloc[0]
    return f'r={r["r"]:+.3f}, p={r["p"]:.3f} {r["sig"]}'

def _get_thresh_str(df_, thresh, metric):
    rows = df_[(df_['threshold'] == thresh) & (df_['metric'] == metric)]
    if len(rows) == 0:
        return 'N/A (n too small)'
    r = rows.iloc[0]
    return f'r={r["r"]:+.3f}, p={r["p"]:.3f} {r["sig"]} (n={r["n_trt"]}T)'

print('Helper functions defined.')

In [ ]:
# ── Summary comparison table ───────────────────────────────────────────────────
print('=' * 90)
print('SUMMARY: Our Results vs. Podolak et al. 2024')
print('=' * 90)
print()
print(f'{"Analysis":<35} {"Podolak 2024":^25} {"Our experiment":^25}')
print('-' * 90)

# Main effect (Views)
views_all = threshold_df[(threshold_df['metric'] == 'Views') & (threshold_df['threshold'] == 0)]
if len(views_all):
    r_all = views_all.iloc[0]
    our_main = f'r={r_all["r"]:+.3f}, p={r_all["p"]:.3f} {r_all["sig"]}'
else:
    our_main = 'see main analysis'

rows_table = [
    ('Main effect (Views)',
     'Suppression >20% (p<0.05)',
     f'Anti-suppression; {our_main}'),
    ('E(i) = Δlikes / (Δviews+1)',
     'Treatment E < Control E (quality drops)',
     f'ctrl={np.median(ctrl_e):.4f}, trt={np.median(trt_e):.4f}, r={eff_e["r"]:+.3f} {eff_e["sig"]}'),
    ('R(i) adjusted (organic replies/view)',
     '~+158% reply rate for original tweets',
     f'ctrl={np.median(ctrl_r_adj):.4f}, trt={np.median(trt_r_adj):.4f}, r={eff_r_adj["r"]:+.3f} {eff_r_adj["sig"]}'),
    ('Tweet type: original post',
     'Suppression significant',
     _get_subgroup_str(subgroup_df, 0, 'Views')),
    ('Tweet type: reply/comment',
     'No significant effect (or positive)',
     _get_subgroup_str(subgroup_df, 1, 'Views')),
    ('Threshold ≥10 baseline views',
     'Effect holds (suppression)',
     _get_thresh_str(threshold_df, 10, 'Views')),
    ('Threshold ≥100 baseline views',
     'Effect disappears',
     _get_thresh_str(threshold_df, 100, 'Views')),
]

for analysis, podolak, ours in rows_table:
    print(f'  {analysis:<33} {podolak:<25} {ours}')
    print()

print()
print('Legend: r < 0 = anti-suppression; r > 0 = suppression; * p<0.05; ** p<0.01; *** p<0.001')
print('        Podolak 2024: LLM-generated CNs, hate speech target, Polish Twitter')
print('        Our study:    Human-crafted CNs, propaganda target, global English/Russian Twitter')